# Solutions · Chapter 03-05 · Bayes' rule you can do on paper

E8 is the one to attempt before reading - it answers a question the chapter raises and does not
settle, and the answer is more alarming than the chapter's version.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def to_odds(p):
    return p / (1 - p)


def to_probability(odds):
    return odds / (1 + odds)


base_rate, sensitivity, false_positive_rate = 1 / 2000, 0.99, 0.01
lr_positive = sensitivity / false_positive_rate
lr_negative = (1 - sensitivity) / (1 - false_positive_rate)
print("LR positive %.1f   LR negative %.4f" % (lr_positive, lr_negative))

## E1 · Naming the three pieces

- **Prior** - `P(has the condition)` = 1/2000, before any test.
- **Likelihood** - `P(positive | has it)` = 0.99, the test's detection rate.
- **Posterior** - `P(has it | positive)` = 0.0472, what you want.

**The manufacturer can tell you the likelihood** - both of them, in fact: the detection rate and the
false-positive rate are properties of the device, measured in the lab.

**They cannot tell you the prior**, because it is not a property of the test at all. It depends on
who is being tested, and it changes between a screening programme and a specialist clinic. And since
they cannot supply the prior, **they cannot supply the posterior either** - which is the number on
the letter, and the number the patient cares about.

## E2 · Why odds are easier

Because updating is a **multiplication** and nothing else. In probability form, updating means
recomputing the denominator `P(E)` every time, which is the sum over all hypotheses. In odds form the
denominator cancels - it appears identically above and below - so the update collapses to
`odds x likelihood ratio`, and repeated evidence is just repeated multiplication.

## E3 · When you may multiply likelihood ratios

**Only when the pieces of evidence are independent *given the hypothesis*** - that is, when knowing
the first result tells you nothing about the second among people who have the condition, and nothing
among people who do not.

Note the phrase "given the hypothesis". Two positive tests are obviously correlated overall, since
both are more likely in someone who is ill. The requirement is that no correlation remains *after*
accounting for that - and the chapter's model B is exactly the case where it does.

## E4 · A different test, done both ways

In [ ]:
people, rate, detect, false_alarm = 10_000, 0.01, 0.80, 0.05
ill = people * rate
well = people - ill
true_positives = ill * detect
false_positives = well * false_alarm

print(pd.DataFrame(
    {"positive": [true_positives, false_positives],
     "negative": [ill - true_positives, well - false_positives]},
    index=["has it", "does not"]).to_string())
print()
print("by counting : %.0f / %.0f = %.4f"
      % (true_positives, true_positives + false_positives,
         true_positives / (true_positives + false_positives)))

odds = to_odds(rate) * (detect / false_alarm)
print("by odds     : %.6f x %.1f = %.4f -> p = %.4f"
      % (to_odds(rate), detect / false_alarm, odds, to_probability(odds)))

**0.1391 both ways.** A positive result takes 1% to 14% - a fourteenfold increase that still leaves
"probably not" as the answer.

Note how much weaker this test is than the chapter's: a likelihood ratio of 16 against 99. And note
that it still helps a great deal - the mistake would be to dismiss a test because a positive does not
settle the question.

## E5 · Positive, positive, negative

In [ ]:
odds = to_odds(base_rate)
print("start          : odds %.6f   p %.6f" % (odds, to_probability(odds)))
for result, ratio in [("positive", lr_positive), ("positive", lr_positive), ("negative", lr_negative)]:
    odds *= ratio
    print("after %-8s : odds %.6f   p %.6f" % (result, odds, to_probability(odds)))

print()
print("product of the three ratios: %.1f x %.1f x %.4f = %.4f"
      % (lr_positive, lr_positive, lr_negative, lr_positive * lr_positive * lr_negative))

**Back to 0.047188** - exactly where one positive left it.

**Does order matter? No**, and the last line is why: the posterior odds are the prior odds times the
*product* of the likelihood ratios, and multiplication commutes. Positive-positive-negative,
negative-positive-positive and positive-negative-positive all give the same answer.

The negative result here has a ratio of 0.0101, almost exactly `1/99`, so it cancels one positive
precisely. That is a coincidence of these particular error rates, not a general rule - but it makes
the arithmetic satisfying: two positives and a negative leave you exactly where one positive did.

## E6 · How strong would a single test have to be?

In [ ]:
prior_odds = to_odds(base_rate)
for target in [0.50, 0.95, 0.99]:
    needed = to_odds(target) / prior_odds
    print("to reach %2.0f%% from a 1-in-2000 prior, a single test needs LR = %.0f"
          % (100 * target, needed))

**1,999 to reach 50%, and 37,981 to reach 95%.**

For comparison, our test has a likelihood ratio of 99. To reach 95% on a single result it would need
a false-positive rate of about **0.0026%** - one in thirty-eight thousand - while keeping the same
detection rate.

**That is the honest reason population screening for rare conditions relies on confirmatory testing
rather than on better single tests.** The required specificity is not achievable, and the arithmetic
says so before any engineering is attempted. A test's job in a screening programme is to reduce a
population of two million to a shortlist of a thousand, which it does superbly. It was never going to
give an answer on its own.

## E7 · `update`

In [ ]:
def update(prior, likelihood_ratios):
    odds = to_odds(prior)
    for ratio in likelihood_ratios:
        odds *= ratio
    return to_probability(odds)


print("three positives      : %.4f" % update(base_rate, [lr_positive] * 3))
print("positive x2, negative: %.6f" % update(base_rate, [lr_positive, lr_positive, lr_negative]))
print("one positive         : %.6f" % update(base_rate, [lr_positive]))
print()
print("E5 check - the last two agree:",
      np.isclose(update(base_rate, [lr_positive, lr_positive, lr_negative]),
                 update(base_rate, [lr_positive])))

## E8 · How much correlation does it take to ruin a confirmatory test?

In [ ]:
sweep_rng = np.random.default_rng(1)
n_people = 4_000_000
truly_ill = sweep_rng.random(n_people) < base_rate

rows = []
for share in [0.0, 0.0005, 0.001, 0.0025, 0.005, 0.0075, 0.01]:
    # `share` of healthy people always test positive. The rest err independently
    # at a rate chosen so the OVERALL false-positive rate stays exactly 1%.
    independent_rate = (false_positive_rate - share) / (1 - share)
    has_trait = sweep_rng.random(n_people) < share

    def run_test():
        return np.where(truly_ill,
                        sweep_rng.random(n_people) < sensitivity,
                        np.where(has_trait, True, sweep_rng.random(n_people) < independent_rate))

    first, second = run_test(), run_test()
    both = first & second
    rows.append({
        "share of healthy with the trait": share,
        "realised false-positive rate": round(float(first[~truly_ill].mean()), 4),
        "P(ill | two positives)": round(float((truly_ill & both).sum() / both.sum()), 4),
    })

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(100 * sweep["share of healthy with the trait"], sweep["P(ill | two positives)"],
        "o-", color="#0072B2")
ax.axhline(0.0472, color="#D55E00", linestyle="--", linewidth=1)
ax.text(0.42, 0.075, "what one positive alone gives", color="#D55E00", fontsize=9)
ax.set_xlabel("share of healthy people whose false positive is stable (%)")
ax.set_ylabel("P(has it | two positives)")
ax.set_title("The false-positive rate is 1% in every point on this line")
plt.tight_layout()
plt.show()

### The shape is the answer, and it is worse than the chapter suggested

**The realised false-positive rate is 0.0100 at every point on that curve.** Nothing measurable about
the test changes. And `P(ill | two positives)` falls from **0.8225 to 0.0468**.

The shape is a steep decay, and **almost all of the damage happens at the left-hand end**:

| Share of healthy people with a stable trait | P(ill after two positives) |
|---|---|
| 0% - all errors independent | 0.8225 |
| 0.05% - a twentieth of the errors | 0.4472 |
| 0.1% - a tenth | 0.3136 |
| 0.25% - a quarter | 0.1609 |
| 1% - all of them | 0.0468 |

**A twentieth of the false positives being correlated already halves the confirmatory value, and a
tenth cuts it to a third.** You do not need a pathological case; you need a small stubborn minority, which is what
biology, sensors and software all reliably provide.

**The general lesson, which is the most useful thing in this chapter:** when you combine evidence,
the result is dominated not by the average quality of the evidence but by **the fraction of it that
is correlated**. The same holds for ensembles of models, redundant sensors and multiple reviewers -
a small shared blind spot bounds how much confidence any amount of agreement can produce.

## E9 · The spam filter

In [ ]:
prior_spam = 0.4
print("three different words, ratios 8, 5, 0.3")
print("  posterior: %.4f" % update(prior_spam, [8, 5, 0.3]))
print()
print("the same word three times, treated as independent")
print("  posterior: %.4f" % update(prior_spam, [8, 8, 8]))

Three different words give **0.8889**. The same word three times gives **0.9971**.

**Why the second is not credible:** the three occurrences are not three independent pieces of
evidence. A message that uses a word once is very likely to use it again - the second and third
occurrences are almost guaranteed by the first, so they carry almost no information beyond it. The
likelihood ratio of 8 was measured for *"contains this word"*, not for *"contains this word a third
time"*.

The right treatment is to use each word's presence once, which is what most practical spam filters do
(a "Bernoulli" model, as opposed to counting occurrences). Any filter that multiplies per occurrence
becomes wildly overconfident on repetitive text - which is why a message consisting of one word
repeated fifty times gets a probability of 0.99999 from a naive implementation, and why that is the
implementation's opinion rather than a fact about the message.

## E10 · What "87% probability of churn" would have to mean

**For that number to mean what a probability should mean, the model must be calibrated:** among all
the customers to whom the model assigns roughly 87%, roughly 87% should actually churn. That is the
definition, and it is the only one that makes the number usable in a decision - if you are deciding
whether a retention offer worth 50 EUR is justified, you are multiplying by that probability, and a
number that is merely *ranked* correctly will give the wrong answer.

**How to check it on historical data:** take a held-out period, bucket the predictions - 0 to 10%,
10 to 20%, and so on - and for each bucket plot the *predicted* average against the *observed* churn
rate. A calibrated model lies on the diagonal. This is a reliability diagram, and it takes about ten
lines.

Two things it commonly reveals: models trained with class weights or resampling are usually
systematically overconfident, and models are often well calibrated in the middle and badly calibrated
at the extremes - which is exactly where the confident decisions are made. Module 07 does this
properly.

## E11 · The security scanner

In [ ]:
threat_rate = 1 / 5_000_000
flag_rate = 1 / 400

for detection in [1.00, 0.90]:
    posterior = detection * threat_rate / flag_rate
    print("if the scanner catches %.0f%% of threats: P(threat | flagged) = %.2e  = 1 in %.0f"
          % (100 * detection, posterior, 1 / posterior))

print()
per_million = 1_000_000 * flag_rate
print("per million passengers: %.0f flags, %.0f minutes of investigation, %.0f staff-hours"
      % (per_million, per_million * 4, per_million * 4 / 60))
print("expected genuine threats among those flags: %.1f" % (1_000_000 * threat_rate))

**About 1 in 12,500** - and per million passengers that is 2,500 flags, 167 staff-hours, and an
expected 0.2 genuine threats.

**What that implies about how the system should be judged.** Not by its precision, which will always
be terrible and cannot be otherwise: the base rate is 1 in 5 million, so even a perfect detector with
a 1-in-400 flag rate produces 12,499 false alarms per real one. Judging it on precision would
condemn any possible system.

It should be judged on:

- **Recall and the cost of a miss.** If the consequence of a missed threat is catastrophic and
  irreversible, a very low precision is rational - this is a case where the asymmetry genuinely
  justifies it.
- **The cost of the false positives**, counted honestly and in full: 167 staff-hours and 2,500
  inconvenienced passengers per million, and whether those costs fall evenly across passengers or
  concentrate on some groups.
- **Whether the alternative uses of that resource would prevent more harm.**

And it should be described honestly to the people operating it. Staff who are told the scanner is
accurate, and who then see thousands of false alarms, stop believing it - which is how the one real
flag gets waved through. Telling them the true rate is what makes the system work.

## E12 · Probabilities of 0.99999 and nothing in between

**The cause is multiplying many correlated likelihood ratios**, which is E9's problem at scale.

Naive Bayes treats every word as independent evidence given the class. Real text is nothing like
that: "free" and "offer", "click" and "here", "prince" and "inheritance" travel together. When forty
correlated words each contribute a ratio of 5, the model multiplies as though it had forty
independent confirmations and produces odds of 5^40.

**The assumption named:** the "naive" in naive Bayes *is* the conditional-independence assumption -
that features are independent given the class. It is known to be false for text and the classifier is
used anyway, for a reason worth understanding: **the ranking survives even though the probabilities
do not.** Multiplying correlated evidence inflates confidence without usually changing which side of
the boundary a message falls on, so accuracy stays good while the probabilities become useless.

The fix, if the probabilities are needed, is calibration after the fact rather than a better model -
fit a simple one-dimensional mapping from the model's scores to observed frequencies. Module 06
covers this.

## E13 · The interview answer

> "A 95% accurate test on a condition affecting 1 in 1,000 gives roughly a 2% chance of actually
> having it. I get that by counting rather than by formula: out of 100,000 people, about 100 have it
> and 95 of those test positive, while 99,900 do not and about 5,000 of them also test positive - so
> 95 out of roughly 5,100, which is under 2%. The number to worry about is the base rate, not the
> accuracy figure. Before finalising I would ask what '95% accurate' actually refers to, because it
> is ambiguous - it might be the detection rate, the specificity, or an overall accuracy on a
> balanced sample, and those three give quite different answers. And I would ask why this person was
> tested, because if they were tested for a reason then the relevant prior is not the population
> rate."

The last two sentences are what distinguishes a good answer. "95% accurate" is genuinely ambiguous,
and the prior depends on who is being tested.

## E14 · Two fraud models

In [ ]:
fraud_prior = 0.003
print("prior                          : %.4f" % fraud_prior)
print("model A alone (LR 40)          : %.4f" % update(fraud_prior, [40]))
print("both models, multiplied (40x35): %.4f" % update(fraud_prior, [40, 35]))

**Naively combined: 0.8082**, against 0.1074 for one model alone.

**What you would need to know before believing it: whether the two models make independent errors
given the truth.** Separately trained is not the same as independent. They almost certainly share:

- **the same training data**, so the same mislabelled examples and the same historical blind spots;
- **the same features**, so a transaction that looks unusual in the feature space looks unusual to
  both;
- **the same definition of fraud**, which is itself a label produced by a process with its own biases.

**What I would expect the true answer to be: substantially below 0.81, and plausibly not far above
the 0.11 that one model gives.** The E8 curve is the guide - it takes only a modest shared blind spot
to remove most of the value of the second opinion.

**How to settle it rather than argue about it:** on held-out data, compute the false-positive rate of
model B *among the transactions model A already flagged incorrectly*. If it equals B's overall
false-positive rate, the errors are independent and you may multiply. If it is much higher - which is
the usual finding - you have measured the correlation directly and can use the measured conditional
rate instead of the marginal one.

That check is four lines and it converts an assumption into a number.

## E15 · For someone non-technical

> "Repeating a test only helps if the test could go wrong for a different reason the second time.
> Sometimes a wrong result is bad luck - a contaminated sample, a machine having an off day - and
> running it again gives an genuinely fresh chance to get it right. But sometimes the wrong result
> comes from something about you that has not changed: your body produces a substance that this
> particular test reacts to. Then the test will say the same thing every time, however often you
> repeat it, and ten identical results are worth barely more than one. A second opinion means a
> different test, not the same test again."

92 words, and it gives the distinction rather than the reassurance, which is what makes it useful.

## E16 · Three hypotheses

In [ ]:
faults = ["brake", "gear", "frame"]
prior = np.array([0.5, 0.3, 0.2])
grinding = np.array([0.70, 0.40, 0.05])          # P(grinding | each fault)
slipping = np.array([0.20, 0.60, 0.30])          # P(slipping  | each fault)

after_grinding = prior * grinding
denominator = after_grinding.sum()
after_grinding = after_grinding / denominator

after_both = after_grinding * slipping
after_both = after_both / after_both.sum()

print(pd.DataFrame({"fault": faults,
                    "prior": prior,
                    "after 'grinding'": after_grinding.round(4),
                    "after 'grinding' and 'slipping'": after_both.round(4)}).to_string(index=False))
print()
print("sums: %.4f and %.4f" % (after_grinding.sum(), after_both.sum()))
print("the denominator after the first symptom was %.4f" % denominator)

After "grinding": brake **0.7292**, gear **0.2500**, frame **0.0208**. Both sum to 1.

After "slipping" as well the picture reverses into a near tie - brake **0.4828** against gear
**0.4966** - because slipping is three times more likely with a gear fault, which almost exactly
undoes grinding's advantage for brakes. Frame stays negligible at 0.0207.

**What the denominator is doing: making the answers sum to 1.** It is `P(evidence)` -
here 0.4800, the total probability of hearing grinding at all, summed across every fault weighted by
how likely that fault was. Dividing by it converts relative weights into probabilities.

Two things worth taking from this:

- **The denominator carries no information about which hypothesis wins.** It is the same for all
  three, so the *ranking* is decided entirely by `prior x likelihood`. This is why the odds form can
  ignore it, and why many practical methods compute only the numerator and normalise at the end.
- **The hypotheses must be exhaustive.** These three probabilities sum to 1 because we asserted the
  fault is one of the three. If the real fault were a loose mudguard, the arithmetic would still
  return a confident answer among brake, gear and frame - one of the more common ways Bayesian
  reasoning produces a confidently wrong result, and one no amount of evidence corrects.

## Where to go next

**03-06 · Functions, lines, slopes, logarithms.** Module 03 leaves uncertainty behind and turns to
the shapes models are made of. It is short, and it is the groundwork for 03-07 and 03-08, after which
you will be able to read what fitting a model actually does.